# Connect kubectl to a Remote k3d Cluster

A self-contained utility for connecting `kubectl` on this Mac to a **k3d cluster already running on a remote server**. It never creates or deletes the cluster: it rebinds the cluster to the server's current IP if that has changed, fetches the kubeconfig, rewrites the server address, saves it locally, verifies the connection, and merges it into `~/.kube/config` so `kubectl` in any terminal can use it. Run the **Disconnect** cell at the end when you are done; it undoes both the saved file and the merge.

## What this runbook does

| Step | What happens |
|---|---|
| Configuration | Set the server address, username and cluster name once |
| 1 - Prerequisites | Verify `kubectl` and SSH are working from this Mac, and that the cluster exists on the server |
| 2 - Rebind | If the server's IP has changed, move the cluster's load balancer ports and API certificate to the new address (does nothing otherwise) |
| 3 - Connect | Fetch the kubeconfig over SSH, rewrite the server address, save it to `~/.kube/` |
| 4 - Verify | Confirm `kubectl` reaches the cluster and all nodes are Ready |
| 5 - Terminal access | Merge the cluster into `~/.kube/config` as context `<CLUSTER_NAME>-remote`, so `kubectl` outside this notebook can use it |
| Disconnect | Remove the `<CLUSTER_NAME>-remote` context, cluster and user from `~/.kube/config`, delete the kubeconfig file and unset `KUBECONFIG` |

**Prerequisites on the server:** Docker Engine, k3d, openssl, an SSH server, and your public key in `~/.ssh/authorized_keys`.

---

## Configuration - run this first

Every `ssh` and `kubectl` command below is built from the values in the next cell.

| Variable | Set it to |
|---|---|
| `SERVER_USER` | Your login **on the server** (not necessarily your Mac username) |
| `SERVER_IP` | The address your Mac reaches the server on - LAN, Tailscale or WireGuard. When it changes, set the new one here and Step 2 rebinds the cluster to it |
| `CLUSTER_NAME` | The k3d cluster name on the server (default `k8scluster`) |

**Re-run this cell after changing any value or restarting the kernel.** The later cells read from the kernel environment, not from this cell, so an edit without a re-run leaves them on the old value.

In [ ]:
import ipaddress
import os
from pathlib import Path

# ---- Set these three ----
SERVER_USER  = "SERVER_USER"   # your login on the server
SERVER_IP    = "SERVER_IP"     # e.g. 192.168.8.10
CLUSTER_NAME = "k8scluster"   # k3d cluster name on the server
# -------------------------

try:
    ipaddress.ip_address(SERVER_IP)
except ValueError:
    if SERVER_IP == "SERVER_IP":
        raise SystemExit("Set SERVER_IP to the server's address before running this cell.")
    raise SystemExit(f"SERVER_IP '{SERVER_IP}' is not a valid IP address.")

if SERVER_USER == "SERVER_USER":
    raise SystemExit("Set SERVER_USER to your login on the server before running this cell.")

SSH_TARGET = f"{SERVER_USER}@{SERVER_IP}"
# Kubeconfig lives outside the repo so it is never committed.
KUBECONFIG = str(Path.home() / ".kube" / f"{CLUSTER_NAME}-remote.yaml")

os.environ["SERVER_USER"]  = SERVER_USER
os.environ["SERVER_IP"]    = SERVER_IP
os.environ["SSH_TARGET"]   = SSH_TARGET
os.environ["CLUSTER_NAME"] = CLUSTER_NAME
os.environ["KUBECONFIG"]   = KUBECONFIG

print(f"SSH_TARGET : {SSH_TARGET}")
print(f"CLUSTER_NAME: {CLUSTER_NAME}")
print(f"KUBECONFIG  : {KUBECONFIG}")

SSH_TARGET : tshepo@192.168.8.103
CLUSTER_NAME: k8scluster
KUBECONFIG  : /Users/tshepo.ntlhokoa/.kube/k8scluster-remote.yaml


---

## Step 1 - Prerequisites

Checks both sides: `kubectl` on this Mac and SSH access to the server.

**SSH must work without a password prompt.** If it asks for a password, copy your public key first:
```
ssh-copy-id SERVER_USER@SERVER_IP
```

Keep `kubectl` within one minor version of the cluster's k3s version (`kubectl version` shows both).

In [7]:
%%bash
: "${SSH_TARGET:?SSH_TARGET is not set - run the Configuration cell first}"

echo "=== This machine ($(hostname -s)) ==="
echo "--- kubectl ---"
kubectl version --client
echo "--- ssh ---"
ssh -V </dev/null 2>&1

echo ""
echo "=== Server ($SSH_TARGET) ==="
if ! ssh -n -o BatchMode=yes -o ConnectTimeout=10 "$SSH_TARGET" true; then
  echo "SSH failed - diagnosing inline:"
  echo "--- TCP 22 ---"
  nc -zv -w 5 "$SERVER_IP" 22 2>&1 || true
  echo "--- ssh -v (last 15 lines) ---"
  ssh -n -v -o BatchMode=yes -o ConnectTimeout=10 "$SSH_TARGET" true 2>&1 | tail -15
  exit 1
fi

echo "SSH OK"
# No -n: the script below arrives on stdin.
ssh -o BatchMode=yes -o ConnectTimeout=10 "$SSH_TARGET" bash -l -s -- "$CLUSTER_NAME" <<'REMOTE'
CLUSTER_NAME=$1
echo ""
echo "--- k3d ---"
k3d version
echo ""
echo "--- clusters ---"
k3d cluster list
if ! k3d cluster list "$CLUSTER_NAME" >/dev/null 2>&1; then
  echo "ERROR: cluster '$CLUSTER_NAME' does not exist on this server."
  echo "Check CLUSTER_NAME in the Configuration cell."
  exit 1
fi
echo ""
echo "Cluster '$CLUSTER_NAME' found."
REMOTE

=== This machine (Tshepo-Ntlhokoa-MAC) ===
--- kubectl ---
Client Version: v1.34.1
Kustomize Version: v5.7.1
--- ssh ---
OpenSSH_10.3p1, LibreSSL 3.3.6

=== Server (tshepo@192.168.8.103) ===
SSH OK

--- k3d ---
k3d version v5.9.0
k3s version v1.35.5-k3s1 (default)

--- clusters ---
NAME         SERVERS   AGENTS   LOADBALANCER
k8scluster   1/1       3/3      true

Cluster 'k8scluster' found.


---

## Step 2 - Rebind the Cluster to the Current Server IP

The server's IP address can change (for example on a new DHCP lease). Two things on the server stay pinned to the address it had when they were set up, and both break when it moves:

| What | Pinned in | Symptom when stale |
|---|---|---|
| Load balancer ports (`k3d-<CLUSTER_NAME>-serverlb`) | The container's port bindings, e.g. `<old IP>:6443` and `<old IP>:8080` | Docker can't bind an address the host no longer has, so the container restart-loops and Step 4 gets `connection refused` |
| API certificate | `tls-san` in `/etc/rancher/k3s/config.yaml` inside `k3d-<CLUSTER_NAME>-server-0` | `kubectl` rejects the certificate (`x509: certificate is valid for ..., not SERVER_IP`) |

This cell checks both against `SERVER_IP` and fixes only what is stale, **in place**: no `k3d cluster delete`, so workloads and data survive. When nothing is stale it changes nothing, so it is safe to run every time.

- **Certificate:** appends `SERVER_IP` to `tls-san` (old entries are kept) and restarts `server-0`. The API is down for about 30 seconds; pods on the agents keep running.
- **Load balancer:** rebuilds the container from its own settings (image, k3d labels, network, volume, nginx `values.yaml`) with every non-loopback port moved to `SERVER_IP`. The old container is kept until the new one starts, and is put back if the rebuild fails.

To stop the address moving, reserve it for the server in the router's DHCP settings.

In [8]:
%%bash
set -euo pipefail
: "${SSH_TARGET:?SSH_TARGET is not set - run the Configuration cell first}"

# No -n: the script arrives on stdin. No -l: a login shell sources ~/.bash_logout,
# whose clear_console call fails under set -e and turns a clean exit into exit 1.
ssh -o BatchMode=yes -o ConnectTimeout=10 "$SSH_TARGET" bash -s -- "$SERVER_IP" "$CLUSTER_NAME" <<'REMOTE'
set -euo pipefail
IP=$1
CLUSTER=$2
SRV="k3d-${CLUSTER}-server-0"
LB="k3d-${CLUSTER}-serverlb"
WORK=$(mktemp -d)
trap 'rm -rf "$WORK"' EXIT

if ! grep -qF "inet ${IP}/" <<<"$(ip -4 -o addr show)"; then
  echo "ERROR: ${IP} is not an address on this server. Its addresses are:"
  ip -4 -br addr
  exit 1
fi
for c in "$SRV" "$LB"; do
  if ! docker inspect "$c" >/dev/null 2>&1; then
    echo "ERROR: container $c not found. Containers for cluster '${CLUSTER}':"
    docker ps -a --filter "label=k3d.cluster=${CLUSTER}" --format 'table {{.Names}}\t{{.Status}}'
    exit 1
  fi
done

cert_sans() {
  local srv_ip
  srv_ip=$(docker inspect "$SRV" --format '{{range .NetworkSettings.Networks}}{{.IPAddress}}{{end}}')
  openssl s_client -connect "${srv_ip}:6443" </dev/null 2>/dev/null \
    | openssl x509 -noout -ext subjectAltName 2>/dev/null \
    | tr ',' '\n' | sed 's/^ *//' || true
}
cert_has_ip() { grep -qxF "IP Address:${IP}" <<<"$(cert_sans)"; }

echo "=== API certificate ==="
SRV_RESTARTED=false
if cert_has_ip; then
  echo "OK: certificate already covers ${IP}"
else
  echo "Certificate does not cover ${IP} - adding it to tls-san"
  docker cp "$SRV:/etc/rancher/k3s/config.yaml" "$WORK/config.yaml" 2>/dev/null || : > "$WORK/config.yaml"
  if ! grep -qxF "  - ${IP}" "$WORK/config.yaml"; then
    if grep -q '^tls-san:' "$WORK/config.yaml"; then
      sed -i "/^tls-san:/a\\  - ${IP}" "$WORK/config.yaml"
    else
      printf 'tls-san:\n  - %s\n' "$IP" >> "$WORK/config.yaml"
    fi
  fi
  docker cp "$WORK/config.yaml" "$SRV:/etc/rancher/k3s/config.yaml"
  echo "--- /etc/rancher/k3s/config.yaml ---"
  cat "$WORK/config.yaml"
  echo "Restarting $SRV ..."
  docker restart "$SRV" >/dev/null
  SRV_RESTARTED=true
  for _ in $(seq 1 60); do
    if docker exec "$SRV" kubectl get --raw=/readyz >/dev/null 2>&1 && cert_has_ip; then
      break
    fi
    sleep 2
  done
  if ! cert_has_ip; then
    echo "ERROR: certificate still does not cover ${IP} after the restart."
    echo "--- SANs now ---"
    cert_sans
    echo "--- $SRV logs (last 20) ---"
    docker logs --tail 20 "$SRV" 2>&1
    exit 1
  fi
  echo "OK: certificate now covers ${IP}"
fi

echo ""
echo "=== Load balancer ports ==="
BINDINGS=$(docker inspect "$LB" --format '{{range $p, $b := .HostConfig.PortBindings}}{{range $b}}{{$p}}|{{.HostIp}}|{{.HostPort}}{{"\n"}}{{end}}{{end}}')
awk -F'|' 'NF { printf "  %s:%s -> %s\n", ($2 == "" ? "0.0.0.0" : $2), $3, $1 }' <<<"$BINDINGS"
STALE=$(awk -F'|' -v ip="$IP" 'NF && $2 != "" && $2 != "0.0.0.0" && $2 != "127.0.0.1" && $2 != ip' <<<"$BINDINGS")

if [ -z "$STALE" ]; then
  echo "OK: no ports bound to an old address"
  if $SRV_RESTARTED; then
    # nginx resolved server-0 when it started; restart it to follow the restarted node.
    docker restart "$LB" >/dev/null
  fi
else
  echo "Ports bound to an old address - rebuilding $LB on ${IP}"
  IMAGE=$(docker inspect "$LB" --format '{{.Config.Image}}')
  RESTART_POLICY=$(docker inspect "$LB" --format '{{.HostConfig.RestartPolicy.Name}}')
  args=(--name "$LB"
        --hostname "$(docker inspect "$LB" --format '{{.Config.Hostname}}')"
        --network "$(docker inspect "$LB" --format '{{.HostConfig.NetworkMode}}')"
        --restart "${RESTART_POLICY:-unless-stopped}")
  while IFS= read -r label; do
    args+=(--label "$label")
  done < <(docker inspect "$LB" --format '{{range $k, $v := .Config.Labels}}{{$k}}={{$v}}{{"\n"}}{{end}}' | grep -E '^(app=|k3d\.)')
  while IFS= read -r env; do
    args+=(-e "$env")
  done < <(comm -23 \
      <(docker inspect "$LB" --format '{{range .Config.Env}}{{.}}{{"\n"}}{{end}}' | sort -u) \
      <(docker image inspect "$IMAGE" --format '{{range .Config.Env}}{{.}}{{"\n"}}{{end}}' | sort -u) | grep .)
  while IFS= read -r mount; do
    args+=(-v "$mount")
  done < <(docker inspect "$LB" --format '{{range .Mounts}}{{if eq .Type "volume"}}{{.Name}}:{{.Destination}}{{"\n"}}{{end}}{{end}}' | grep .)
  while IFS='|' read -r port hip hport; do
    case "$hip" in
      127.0.0.1)  args+=(-p "127.0.0.1:${hport}:${port}") ;;
      ""|0.0.0.0) args+=(-p "${hport}:${port}") ;;
      *)          args+=(-p "${IP}:${hport}:${port}") ;;
    esac
  done < <(grep . <<<"$BINDINGS")

  docker cp "$LB:/etc/confd/values.yaml" "$WORK/values.yaml"
  docker stop "$LB" >/dev/null
  docker rename "$LB" "${LB}-old"
  if docker create "${args[@]}" "$IMAGE" >/dev/null \
     && docker cp "$WORK/values.yaml" "$LB:/etc/confd/values.yaml" \
     && docker start "$LB" >/dev/null; then
    docker rm "${LB}-old" >/dev/null
    echo "Rebuilt $LB"
  else
    echo "ERROR: rebuilding $LB failed - putting the previous container back"
    docker rm -f "$LB" >/dev/null 2>&1 || true
    docker rename "${LB}-old" "$LB"
    docker start "$LB" >/dev/null 2>&1 || true
    exit 1
  fi
fi

echo ""
echo "=== ${IP}:6443 ==="
for _ in $(seq 1 15); do
  if timeout 2 bash -c "</dev/tcp/${IP}/6443" 2>/dev/null; then
    echo "OK: ${IP}:6443 accepts connections"
    docker port "$LB"
    exit 0
  fi
  sleep 1
done
echo "ERROR: nothing accepts connections on ${IP}:6443"
docker ps -a --filter "name=^${LB}\$" --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}'
echo "--- $LB logs (last 20) ---"
docker logs --tail 20 "$LB" 2>&1
exit 1
REMOTE

=== API certificate ===
OK: certificate already covers 192.168.8.103

=== Load balancer ports ===
  127.0.0.1:5432 -> 5432/tcp
  192.168.8.103:6443 -> 6443/tcp
  192.168.8.103:8080 -> 80/tcp
OK: no ports bound to an old address

=== 192.168.8.103:6443 ===
OK: 192.168.8.103:6443 accepts connections
80/tcp -> 192.168.8.103:8080
5432/tcp -> 127.0.0.1:5432
6443/tcp -> 192.168.8.103:6443


---

## Step 3 - Fetch the Kubeconfig

Asks k3d on the server for the kubeconfig, rewrites the `server:` line to `https://SERVER_IP:6443`, and saves it to `~/.kube/<CLUSTER_NAME>-remote.yaml` with permissions `0600`. The rewrite is required: k3d always reports the address the cluster was created with (here `https://192.168.10.152:6443`), and a rebind in Step 2 cannot change that.

The file streams straight over SSH into a local file - **no copy is left on the server**. It holds cluster-admin credentials: keep it out of git.

In [10]:
%%bash
set -euo pipefail
: "${SSH_TARGET:?SSH_TARGET is not set - run the Configuration cell first}"
: "${KUBECONFIG:?KUBECONFIG is not set - run the Configuration cell first}"

mkdir -p "$(dirname "$KUBECONFIG")"
umask 077

# Captured into a variable first so a failed ssh or k3d aborts (set -e) before
# the existing kubeconfig is overwritten with an empty file.
RAW=$(ssh -n -o BatchMode=yes -o ConnectTimeout=10 "$SSH_TARGET" \
  "bash -lc 'k3d kubeconfig get ${CLUSTER_NAME}'")
printf '%s\n' "$RAW" \
  | sed "s#server: https://.*#server: https://${SERVER_IP}:6443#" \
  > "$KUBECONFIG"
chmod 600 "$KUBECONFIG"

echo "Wrote $KUBECONFIG"
echo ""
echo "Server line in kubeconfig:"
grep 'server:' "$KUBECONFIG"

Wrote /Users/tshepo.ntlhokoa/.kube/k8scluster-remote.yaml

Server line in kubeconfig:
    server: https://192.168.8.103:6443


In [ ]:
%%bash
cp ~/.kube/config ~/.kube/config.bak
KUBECONFIG=~/.kube/config:~/.kube/k8scluster-remote.yaml kubectl config view --flatten > /tmp/merged \
  && mv /tmp/merged ~/.kube/config && chmod 600 ~/.kube/config


---

## Step 4 - Verify the Connection

Confirms that `kubectl` on this Mac reaches the cluster API and that all nodes are Ready.

In [11]:
%%bash
: "${KUBECONFIG:?KUBECONFIG is not set - run the Configuration cell first}"

if ! kubectl cluster-info 2>&1; then
  echo ""
  echo "kubectl cannot reach the API server - diagnosing inline:"
  echo "--- TCP ${SERVER_IP}:6443 ---"
  nc -zv -w 5 "$SERVER_IP" 6443 2>&1 || true
  echo "--- addresses in the API certificate (SERVER_IP must be listed) ---"
  openssl s_client -connect "${SERVER_IP}:6443" </dev/null 2>/dev/null \
    | openssl x509 -noout -text 2>/dev/null \
    | grep -A1 "Subject Alternative Name" \
    || echo "(no certificate present or openssl not installed)"
  echo ""
  echo "A refused connection, or a SAN list without ${SERVER_IP}, means the cluster"
  echo "is still bound to an old server IP. Step 2 rebinds it to ${SERVER_IP}."
  exit 1
fi

echo ""
echo "=== Nodes ==="
kubectl wait --for=condition=Ready node --all --timeout=60s >/dev/null
kubectl get nodes -o wide
echo ""
echo "kubectl is connected to the remote cluster."

Kubernetes control plane is running at https://192.168.8.103:6443
CoreDNS is running at https://192.168.8.103:6443/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy
Metrics-server is running at https://192.168.8.103:6443/api/v1/namespaces/kube-system/services/https:metrics-server:https/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.

=== Nodes ===
NAME                      STATUS   ROLES           AGE   VERSION        INTERNAL-IP   EXTERNAL-IP   OS-IMAGE           KERNEL-VERSION     CONTAINER-RUNTIME
k3d-k8scluster-agent-0    Ready    <none>          13d   v1.35.3+k3s1   172.18.0.2    <none>        K3s v1.35.3+k3s1   7.0.0-42-generic   containerd://2.2.2-k3s1
k3d-k8scluster-agent-1    Ready    <none>          13d   v1.35.3+k3s1   172.18.0.3    <none>        K3s v1.35.3+k3s1   7.0.0-42-generic   containerd://2.2.2-k3s1
k3d-k8scluster-agent-2    Ready    <none>          13d   v1.35.3+k3s1   172.18.0.4    <none>        K3s v1.35.3+k3s1   7.0

---

## Disconnect

Undoes Steps 3 and 5. Run this when you are finished with the cluster:

- Removes the `<CLUSTER_NAME>-remote` context, cluster and user from `~/.kube/config`. If that context was the current one, the current context is unset, because there is no record of which context you used before it.
- Deletes `~/.kube/<CLUSTER_NAME>-remote.yaml` and unsets `KUBECONFIG` in the kernel, so later `kubectl` calls fall back to `~/.kube/config`.

Nothing on the server changes: the cluster keeps running.

In [ ]:
import os
import subprocess
from pathlib import Path

cluster_name = os.environ.get("CLUSTER_NAME", "k8scluster")
merged_name = f"{cluster_name}-remote"
local_config = Path.home() / ".kube" / "config"

# 1. Remove the entries Step 5 merged into ~/.kube/config. --kubeconfig is explicit
#    because KUBECONFIG in this kernel points at the remote file, not ~/.kube/config.
def kc(*args):
    return subprocess.run(["kubectl", "--kubeconfig", str(local_config), "config", *args],
                          capture_output=True, text=True)

if local_config.exists():
    if kc("current-context").stdout.strip() == merged_name:
        kc("unset", "current-context")
        print(f"Unset current-context (it was {merged_name}), so no context is selected now")
    for kind in ("context", "cluster", "user"):
        if kc(f"delete-{kind}", merged_name).returncode == 0:
            print(f"Removed {kind} {merged_name} from {local_config}")
        else:
            print(f"Not present (already removed): {kind} {merged_name} in {local_config}")
else:
    print(f"Not present: {local_config}")

# 2. Remove the kubeconfig file Step 3 wrote.
kubeconfig = Path(os.environ.get("KUBECONFIG", ""))
if not kubeconfig.name:
    # KUBECONFIG unset - derive the path from CLUSTER_NAME as a fallback
    kubeconfig = Path.home() / ".kube" / f"{cluster_name}-remote.yaml"

if kubeconfig.exists():
    kubeconfig.unlink()
    print(f"Removed {kubeconfig}")
else:
    print(f"Not present (already removed): {kubeconfig}")

os.environ.pop("KUBECONFIG", None)
print("KUBECONFIG unset - kubectl will use ~/.kube/config from now on.")

---

## Disconnect

Removes the kubeconfig file and unsets `KUBECONFIG` in the kernel so subsequent `kubectl` calls fall back to `~/.kube/config`. Run this when you are finished with the cluster.

In [5]:
import os
from pathlib import Path

kubeconfig = Path(os.environ.get("KUBECONFIG", ""))
if not kubeconfig.name:
    # KUBECONFIG unset - derive the path from CLUSTER_NAME as a fallback
    cluster_name = os.environ.get("CLUSTER_NAME", "k8scluster")
    kubeconfig = Path.home() / ".kube" / f"{cluster_name}-remote.yaml"

if kubeconfig.exists():
    kubeconfig.unlink()
    print(f"Removed {kubeconfig}")
else:
    print(f"Not present (already removed): {kubeconfig}")

os.environ.pop("KUBECONFIG", None)
print("KUBECONFIG unset - kubectl will use ~/.kube/config from now on.")

Removed /Users/tshepo.ntlhokoa/.kube/k8scluster-remote.yaml
KUBECONFIG unset - kubectl will use ~/.kube/config from now on.
